# 07.2 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/07-forecasting/02-baselines-and-evaluation/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

# The two sheets overlap: "Year 2009-2010" runs to 9 Dec 2010, "Year 2010-2011" starts 1 Dec 2010.
older = load_retail("Year 2009-2010")
newer = load_retail("Year 2010-2011")
raw = pd.concat([older[older["InvoiceDate"] < "2010-12-01"], newer], ignore_index=True)

def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df[~df["Invoice"].astype(str).str.startswith("C")]
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
    return out.assign(Revenue=out["Quantity"] * out["Price"])

sales = clean_retail(raw)
sales = sales[sales["Quantity"] < 70_000]          # two one-off orders, both cancelled the same day
END = pd.Timestamp("2011-12-04")                    # last complete week (Mon 28 Nov - Sun 4 Dec)
sales = sales[sales["InvoiceDate"] < END + pd.Timedelta(days=1)]

rev = sales.set_index("InvoiceDate")["Revenue"]
daily = rev.resample("D").sum()                     # every calendar day, 0 when closed
weekly = rev.resample("W").sum()                    # weeks end on Sunday
monthly = rev.resample("ME").sum()                  # month-end dates
monthly = monthly[monthly.index <= "2011-11-30"]    # December 2011 has only 4 days: drop it
print(f"{len(sales):,} order lines, {daily.index[0].date()} to {daily.index[-1].date()}")
print(f"{len(daily)} days, {len(weekly)} weeks, {len(monthly)} months")

H = 5                                                   # forecast 5 weeks ahead
ORIGINS = pd.date_range("2011-04-03", periods=8, freq="4W-SUN")      # 3 Apr ... 16 Oct 2011

def backtest(forecast_fn, series=None, origins=ORIGINS, h=H):
    """Rolling-origin backtest: at each origin, fit on the past, forecast h weeks, score MAE."""
    series = weekly if series is None else series
    rows = []
    for origin in origins:
        train = series[series.index <= origin]
        test = series[series.index > origin].iloc[:h]
        pred = np.asarray(forecast_fn(train, len(test)))
        rows.append({"origin": origin.date(), "MAE": np.abs(test.values - pred).mean()})
    return pd.DataFrame(rows)

def naive(train, h):          return np.repeat(train.iloc[-1], h)
def mean_fc(train, h):        return np.repeat(train.mean(), h)
def seasonal_naive(train, h): return train.iloc[-52:-52 + h].values
def drift(train, h):
    slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    return train.iloc[-1] + slope * np.arange(1, h + 1)

def mae(y, p):   return np.mean(np.abs(y - p))
def rmse(y, p):  return np.sqrt(np.mean((y - p) ** 2))
def mape(y, p):
    with np.errstate(divide="ignore", invalid="ignore"):   # 0/0 gives nan, x/0 gives inf
        return np.mean(np.abs(y - p) / np.abs(y)) * 100
def smape(y, p): return np.mean(2 * np.abs(y - p) / (np.abs(y) + np.abs(p))) * 100

768,341 order lines, 2009-12-01 to 2011-12-04
734 days, 105 weeks, 24 months


## Exercise 1

- Naive: 27 for every week → errors 4, 0, 7, 1 → MAE **3.0**.
- Seasonal naive: `[22, 25, 33, 27]` → errors 1, 2, 1, 1 → MAE **1.25**.
- Drift: slope (27 − 20) / 7 = 1 → `[28, 29, 30, 31]` → errors 5, 2, 4, 3 → MAE **3.5**.

In [2]:
tr = np.array([20, 24, 30, 26, 22, 25, 33, 27.]); te = np.array([23, 27, 34, 28.])
fc = {"naive": np.repeat(tr[-1], 4), "seasonal naive": tr[-4:],
      "drift": tr[-1] + (tr[-1] - tr[0]) / (len(tr) - 1) * np.arange(1, 5)}
for k, p in fc.items():
    print(f"{k:<15} {p}  MAE {mae(te, p):.2f}")

naive           [27. 27. 27. 27.]  MAE 3.00
seasonal naive  [22. 25. 33. 27.]  MAE 1.25
drift           [28. 29. 30. 31.]  MAE 3.50


## Exercise 2

In [3]:
y2011 = weekly[weekly.index.year == 2011]
last_year = weekly.shift(52).loc[y2011.index]
print(f"MAE  £{mae(y2011.values, last_year.values):,.0f}")
print(f"MAPE {mape(y2011.values, last_year.values):,.0f}%")
ape = (np.abs(y2011 - last_year) / y2011 * 100).sort_values(ascending=False)
pd.DataFrame({"actual": y2011, "last year": last_year, "APE %": ape}).loc[ape.index[:3]].round(0)

MAE  £29,143
MAPE nan%


,actual,last year,APE %
InvoiceDate,,,
2011-05-01,73374.0,170901.0,133.0
2011-10-02,193654.0,302393.0,56.0
2011-07-03,116251.0,170794.0,47.0


MAPE comes out as `nan`: the week ending 2 January 2011 sold nothing (nor did the same week a year earlier), and 0 / 0 is undefined. One closed week ruins the metric for the whole year, while MAE stays meaningful.

Among the remaining weeks, the largest percentage error is the week ending 1 May 2011 (133%). That week had two UK bank holidays, Easter Monday and the royal-wedding holiday, while the same week of 2010 was an ordinary trading week. "Same week last year" breaks whenever the calendar moves, which is a limitation of seasonal naive worth remembering.

## Exercise 3

In [4]:
res10 = {n: backtest(fn, h=10)["MAE"].mean() for n, fn in
         [("naive", naive), ("mean", mean_fc), ("seasonal naive", seasonal_naive), ("drift", drift)]}
res5 = {n: backtest(fn)["MAE"].mean() for n, fn in
        [("naive", naive), ("mean", mean_fc), ("seasonal naive", seasonal_naive), ("drift", drift)]}
pd.DataFrame({"h=5": res5, "h=10": res10}).round(0)

,h=5,h=10
naive,42200.0,44801.0
mean,49412.0,52981.0
seasonal naive,32450.0,31134.0
drift,43037.0,48413.0


Seasonal naive stays best. Naive and drift get worse at h = 10, because the further ahead you go the more likely the season has turned, and drift keeps extrapolating a straight line across a curve.

## Exercise 4

In [5]:
def seasonal_naive_adjusted(train, h):
    ratio = train.iloc[-8:].sum() / train.iloc[-60:-52].sum()
    return train.iloc[-52:-52 + h].values * ratio

cmp = pd.DataFrame({"seasonal naive": backtest(seasonal_naive)["MAE"],
                    "adjusted": backtest(seasonal_naive_adjusted)["MAE"]}).set_axis([o.date() for o in ORIGINS])
print((cmp.mean() / 1000).round(1).rename("mean MAE £k"))
(cmp / 1000).round(1)

seasonal naive    32.4
adjusted          33.2
Name: mean MAE £k, dtype: float64


,seasonal naive,adjusted
2011-04-03,29.9,20.8
2011-05-01,22.9,37.1
2011-05-29,25.2,26.5
2011-06-26,27.0,31.7
2011-07-24,16.1,15.4
2011-08-21,41.3,39.8
2011-09-18,75.2,72.7
2011-10-16,21.9,21.3


The adjustment helps at some origins and hurts at others, and on average it does not beat plain seasonal naive here. ShopWave 2011 runs at roughly the same level as 2010, so there is little level change to correct, and an 8-week ratio is itself noisy. That is a useful lesson: a sensible idea still has to win the backtest.